In [23]:
import pandas as pd
from functools import reduce
import numpy as np

In [24]:
# Load GO-Gene annotations
GO_annots = pd.read_csv("/space/grp/aadrian/Pseudobulk_Function_Pipeline_HighRes/bin/preprocessing/preprocessGO_pipe/data/2024_march/data/processing/bp_annotations_withGeneData_qc_annotations.csv")

# Differential expression analysis
dea = pd.read_csv("/space/grp/aadrian/Pseudobulk_Function_Pipeline_HighRes/bin/deconvolutingBulk/bin/notebooks/all_gene_marker_scores.csv", index_col=0)

# Calculates CTEs

In [30]:
def calc_unscaled_wilcox_minandmax(filtered_merged, gene)->float:
    """Calculate marker score for one gene in a GO term

    Args:
        filtered_merged (_type_): Df containing log fold changes for genes (hgnc_symbol)

    Returns:
        float: marker score
    """
    # For each gene, get the index of the largest log fold change
    max_gene_id = filtered_merged.scores.idxmax() 
    max_genes_df = filtered_merged.loc[max_gene_id,:] # df of the top single log fold changed genes
    
    is_max = filtered_merged.index.isin([max_gene_id]) # bool mask for if the gene is the max gene or not
    # Select rows that are not the maximum
    min_genes_df = filtered_merged.loc[~is_max] # df of the remaining, not max genes
    # lowest_value =abs(min_genes_df.loc[:,'logfoldchanges'].min())
    
    max_score = max_genes_df.loc['scores'] # max log fold change
    min_score = min_genes_df.loc[:,'scores'].mean() # mean of all of the log fold changes
    
    # marker_score = max_score/min_score
    return pd.DataFrame({'gene':[gene], "wilcox_max_score":[max_score], "wilcox_min_score":[min_score]})

def calc_unscaled_logfc_minandmax(filtered_merged, gene)->float:
    """Calculate marker score for one gene in a GO term

    Args:
        filtered_merged (_type_): Df containing log fold changes for genes (hgnc_symbol)

    Returns:
        float: marker score
    """
    # For each gene, get the index of the largest log fold change
    max_gene_id = filtered_merged.logfoldchanges.idxmax() 
    max_genes_df = filtered_merged.loc[max_gene_id,:] # df of the top single log fold changed genes
    
    is_max = filtered_merged.index.isin([max_gene_id]) # bool mask for if the gene is the max gene or not
    # Select rows that are not the maximum
    min_genes_df = filtered_merged.loc[~is_max] # df of the remaining, not max genes
    # lowest_value =abs(min_genes_df.loc[:,'logfoldchanges'].min())
    
    max_score = max_genes_df.loc['logfoldchanges'] # max log fold change
    min_score = min_genes_df.loc[:,'logfoldchanges'].mean() # mean of all of the log fold changes
    
    # marker_score = max_score/min_score
    return pd.DataFrame({'gene':[gene], "fc_max_score":[max_score], "fc_min_score":[min_score]})

def calc_max_difference_score(filtered_merged, gene)->float:
    """Calculate marker score for one gene in a GO term

    Args:
        filtered_merged (_type_): Df containing log fold changes for genes (hgnc_symbol)

    Returns:
        float: marker score
    """
    # For each gene, get the index of the largest log fold change
    filtered_merged = filtered_merged.sort_values('scores', ascending=False)
    
    ser_select = filtered_merged['scores'].iloc[0:2]
    
    diff_between_1rst_2nd = ser_select.iloc[0] - ser_select.iloc[1]
    print(ser_select.iloc[0])
    print(ser_select.iloc[1])
        
    return pd.DataFrame({"gene":[gene], "max_diff_score":[diff_between_1rst_2nd]})
    
def calc_unscaled_scores(df, gene):    
    lo_unscaled_scores = []
    
    lo_unscaled_scores.append(calc_unscaled_wilcox_minandmax(df, gene))
    lo_unscaled_scores.append(calc_unscaled_logfc_minandmax(df, gene))
    lo_unscaled_scores.append(calc_max_difference_score(df, gene))
    
    df_unscaled_scores = reduce(lambda left,right: pd.merge(left,right, on = 'gene', how = 'left'), lo_unscaled_scores)
    return df_unscaled_scores


def scale_one_marker_score_type(df_all_marker_scores, min_col, max_col):
    scalar = df_all_marker_scores.loc[:,min_col].min()

    df_all_marker_scores[f"{max_col}_scaled"] = df_all_marker_scores.loc[:,max_col] + abs(scalar)+1
    df_all_marker_scores[f"{min_col}_scaled"] = df_all_marker_scores.loc[:,min_col] + abs(scalar)+1
        
    return df_all_marker_scores
    

def scale_marker_scores(df_unscaled_scores):
    # Scale log2fcs
    df_scaled_scores = scale_one_marker_score_type(df_unscaled_scores, min_col = 'fc_min_score', max_col = "fc_max_score")
    # scale wilcoxon scores
    df_scaled_scores = scale_one_marker_score_type(df_unscaled_scores, min_col = 'wilcox_min_score', max_col = "wilcox_max_score")
    return df_scaled_scores

def calc_final_scores(df_scaled_scores):
    df_scaled_scores['wilcox_score'] = df_scaled_scores.loc[:,'wilcox_max_score_scaled']/df_scaled_scores.loc[:,'wilcox_min_score_scaled']
    df_scaled_scores['logfc_score'] = df_scaled_scores.loc[:,'fc_max_score_scaled']/df_scaled_scores.loc[:,'fc_min_score_scaled']
    return df_scaled_scores

def calc_all_gene_enrichment_scores(df_all_scores:pd.DataFrame):
    lo_unscaled_scores = []
    for gene, df in df_all_scores.groupby("names"):
        # Calculate unscaled MGC scores        
        df_one_unscaled_scores = calc_unscaled_scores(df=df, gene=gene)
        lo_unscaled_scores.append(df_one_unscaled_scores)
    # concat all the unscaled scores
    df_unscaled_scores = pd.concat(lo_unscaled_scores, axis = 0)

    return df_unscaled_scores
    
dea_scored = calc_all_gene_enrichment_scores(dea)
dea_scored.head()

6.8800364
-0.5030107
7.3830471
1.4524969
-0.2201268
1.6726237000000002
13.936005
7.96195
5.974055
11.515552
0.076370016
11.439181984
3.3583012
-0.5080346
3.8663358
0.90818
0.322585
0.585595
0.07198619
-0.019043228
0.091029418
15.734682
-2.1600392
17.8947212
99.76272
-14.076869
113.839589
0.11606443
-0.022916425
0.138980855
0.020597452
-0.001295832
0.021893284
0.09899286
-0.012910662
0.111903522
0.25023758
-0.029371757
0.279609337
25.3218
-4.2721715
29.5939715
56.170017
-10.160558
66.330575
49.457413
-4.815096
54.272509
3.8319995
-0.7495667
4.5815662
13.8606205
-4.91557
18.7761905
3.024899
-0.48382708
3.50872608
9.956351
-2.9824321
12.9387831
3.3746936
-0.6219712
3.9966648
16.400284
1.3942494
15.0060346
5.904799
0.35867682
5.546122179999999
13.932295
-2.724465
16.65676
28.168686
-7.0615616
35.2302476
36.906796
18.090712
18.816084
32.518723
-6.3378315
38.8565545
44.748207
2.9806082
41.7675988
90.17977
-2.213177
92.392947
38.41523
5.4597282
32.9555018
26.306557
8.549136
17.757421
3.312342

15.061642
-3.397709
18.459351
0.5215828
0.22789277
0.29369003000000005
15.876239
-4.3503346
20.226573600000002
0.026911413
-0.0016138328
0.028525245799999998
15.250196
5.5185876
9.7316084
20.124052
18.021645
2.1024069999999995
20.056753
-5.4692497
25.5260027
12.751801
9.836358
2.915443
20.455627
17.536676
2.918951
16.256977
-1.7716175
18.0285945
2.2283726
0.022704141
2.205668459
24.253439
-7.816093
32.069532
15.5680895
-0.7359594
16.304048899999998
37.387802
-8.036972
45.424774
42.815434
18.87644
23.938994000000005
98.324356
-7.9000936
106.2244496
0.25600207
0.17182694
0.08417513000000001
16.238192
0.9596818
15.278510200000001
31.987429
-7.3052077
39.2926367
9.142771
-2.6153042
11.7580752
57.298527
-4.749593
62.04812
73.84885
35.67295
38.1759
71.882286
-18.051058
89.93334399999999
13.381231
1.8953656
11.4858654
18.769207
-0.876695
19.645902000000003
49.652752
8.20746
41.445292
0.75174373
-0.077057496
0.8288012260000001
34.65019
-4.775951
39.426141
7.9839783
-1.4823415
9.4663198
31.3572

,gene,wilcox_max_score,wilcox_min_score,fc_max_score,fc_min_score,max_diff_score
0,A1BG,6.880036,-2.270726,0.879714,-11.890932,7.383047
0,A1CF,1.452497,-0.530261,1.082467,-11.338889,1.672624
0,A2M,13.936005,-1.451539,7.711000,-0.801732,5.974055
0,A2ML1,11.515552,-4.485514,0.799550,-7.422299,11.439182
0,A3GALT2,3.358301,-1.241936,1.106296,-16.601308,3.866336


# Calculate MGC Scores

In [26]:
def calc_wilcox_MGC(df_all_marker_scores_filtered):
    return np.mean(df_all_marker_scores_filtered['wilcox_max_score'])
def calc_logfc_MGC(df_all_marker_scores_filtered):
    return  np.mean(df_all_marker_scores_filtered['fc_max_score'])
def get_diff_MGC(df_all_marker_scores_filtered):
    return np.mean(df_all_marker_scores_filtered.loc[:,'max_diff_score'])

def calc_all_MGC_for_go_term(go_term, go_annot, dea):
    # Filter df for genes in GO term
    go_annot_filtered = go_annot[go_annot.loc[:,"GO ID"] == go_term]
    # get genes
    lo_genes_in_go_term = go_annot_filtered.DB_Object_Symbol
    
    # Get marker gene scores for the genes
    df_all_marker_scores_filtered = dea[dea.loc[:,'gene'].isin(lo_genes_in_go_term)]
    
    # Calc MGC types
    wilcox_MGC = calc_wilcox_MGC(df_all_marker_scores_filtered)

    # lofgc MGC
    logfc_MGC = calc_logfc_MGC(df_all_marker_scores_filtered)
    # max diff between 1rst and second MGC
    diff_MGC = get_diff_MGC(df_all_marker_scores_filtered)
    
    ser_MGC = pd.Series([go_term, wilcox_MGC, logfc_MGC, diff_MGC], index = ['id', 'wilcox_MGC', 'logfc_MGC', 'diff_MGC'])
    
    return ser_MGC
    
def calc_all_marker_scores(go_annot:pd.DataFrame, dea:pd.DataFrame) -> pd.DataFrame:
    """wrapper function to calculate the marker scores for each go term
    
    marker score is how much a GO term has genes has pseud-marker genes

    Args:
        enrichment_master (pd.DataFrame): dataframe that has all of our enriched GO terms
        go_annot (pd.DataFrame): df that tells us what genes are affiliated with each go term
        df_all_scores (pd.DataFrame): df contianing wilcoxon tests for 1 vs all for each cell type and each gene


    Returns:
        pd.DataFrame: _description_
    """
    lo_go_terms = go_annot.loc[:,'GO ID'].unique()
    lo_MGC_for_go_term = []
    for go_term in lo_go_terms:
        ser_MGC = calc_all_MGC_for_go_term(go_term=go_term, go_annot=go_annot, dea=dea)
        lo_MGC_for_go_term.append(ser_MGC)
            
    return pd.concat(lo_MGC_for_go_term, axis = 1).T

mgcs = calc_all_marker_scores(go_annot=GO_annots,
                            dea=dea_scored)
mgcs.head()

,id,wilcox_MGC,logfc_MGC,diff_MGC
0,GO:0000045,22.131554,1.317919,25.622232
1,GO:0000070,13.388181,1.29324,14.892069
2,GO:0000077,20.420476,1.287744,23.342727
3,GO:0000079,19.141894,1.487674,20.550454
4,GO:0000082,30.492923,1.632122,32.970738


In [27]:
mgcs.to_csv("data/brain_mgcs.csv")